# Get Failed Files in Corpus

In [109]:
import sys
import re

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import read_txt
from utils import list_xlsx_files

In [110]:
models = ["gpt2"]

corpora = ["Wiki", "Enron", "Perverted Justice", "StackExchange",
            "ACL", "TripAdvisor", "The Apricity", "Koppel's Blogs",
            "The Telegraph", "Reddit"]
corpora = ["Wiki"]

raw_values = ["raw_100", "raw_200", "raw_300", "raw_400", "raw_500",
              "raw_600", "raw_700", "raw_800", "raw_900", "raw_1000"]

data_types = ["training", "test"]

base_expected_loc = "/Volumes/BCross/datasets/author_verification"

base_completed_loc = "/Volumes/BCross/av_datasets_experiments/ngram_masking_logrpobs_n1"

In [111]:
def get_unprocessed_files(problem_loc: str, completed_loc: str):
    """Get the names of missing files from a location and thier position in the problem list"""
    
    def missing_with_positions(expected, actual):
        actual_set = set(actual)  # fast membership checks
        return [(i, item) for i, item in enumerate(expected, start=1) if item not in actual_set]

    # Get the problem list as a string and convert to a list
    problem_list_str = read_txt(problem_loc)
    problem_list = [line for line in problem_list_str.split("\n") if line.strip()]
    
    # Get the completed files by listing all .xlsx files in loc
    completed_file_locs = list_xlsx_files(completed_loc)
    
    # Get file name from path and remove .xlsx
    completed_probs = [cf.name.replace(".xlsx", "") for cf in completed_file_locs]
    
    missing_files = missing_with_positions(problem_list, completed_probs)

    return missing_files

In [112]:
def parse_raw_value(raw_value: str):
    s = str(raw_value).strip()
    if s == "raw":
        return None
    m = re.fullmatch(r"raw_(\d+)", s)
    return int(m.group(1)) if m else None

In [113]:
rows = []

for d_type in data_types:
    for corpus in corpora:
        problem_loc = f"{base_expected_loc}/{d_type}/{corpus}/problem_doc_list.txt"

        for model in models:
            for r_val in raw_values:
                max_context_tokens = parse_raw_value(r_val)
                completed_loc = f"{base_completed_loc}/{d_type}/{corpus}/{model}/{r_val}"

                missing = get_unprocessed_files(problem_loc, completed_loc)

                for expected_position, item in missing:
                    rows.append({
                        "data_type": d_type,
                        "corpus": corpus,
                        "scoring_model": model,
                        "raw_value": r_val,
                        "max_context_tokens": max_context_tokens,
                        "expected_position": expected_position,
                        "item": item,
                    })

df_missing = pd.DataFrame(
    rows,
    columns=["data_type", "corpus", "scoring_model", "raw_value", "max_context_tokens", "expected_position", "item"]
)

In [114]:
df_missing

,data_type,corpus,scoring_model,raw_value,max_context_tokens,expected_position,item
0,test,Wiki,gpt2,raw_100,100,97,jéské_couriano_text_1 vs jéské_couriano_text_4
1,test,Wiki,gpt2,raw_100,100,98,jéské_couriano_text_3 vs jéské_couriano_text_4
2,test,Wiki,gpt2,raw_100,100,99,jéské_couriano_text_5 vs jéské_couriano_text_4
3,test,Wiki,gpt2,raw_100,100,100,jéské_couriano_text_1 vs kashmiri_text_3
4,test,Wiki,gpt2,raw_100,100,101,jéské_couriano_text_3 vs kashmiri_text_3
...,...,...,...,...,...,...,...
85,test,Wiki,gpt2,raw_1000,1000,101,jéské_couriano_text_3 vs kashmiri_text_3
86,test,Wiki,gpt2,raw_1000,1000,102,jéské_couriano_text_5 vs kashmiri_text_3
87,test,Wiki,gpt2,raw_1000,1000,127,jwanders_text_1 vs jéské_couriano_text_4
88,test,Wiki,gpt2,raw_1000,1000,128,jwanders_text_2 vs jéské_couriano_text_4


In [115]:
group_cols = ["data_type", "corpus", "scoring_model", "expected_position", "item"]

out = (
    df_missing
    .groupby(group_cols, as_index=False)["max_context_tokens"]
    .min(min_count=1)
)

out

,data_type,corpus,scoring_model,expected_position,item,max_context_tokens
0,test,Wiki,gpt2,97,jéské_couriano_text_1 vs jéské_couriano_text_4,100
1,test,Wiki,gpt2,98,jéské_couriano_text_3 vs jéské_couriano_text_4,100
2,test,Wiki,gpt2,99,jéské_couriano_text_5 vs jéské_couriano_text_4,100
3,test,Wiki,gpt2,100,jéské_couriano_text_1 vs kashmiri_text_3,100
4,test,Wiki,gpt2,101,jéské_couriano_text_3 vs kashmiri_text_3,100
5,test,Wiki,gpt2,102,jéské_couriano_text_5 vs kashmiri_text_3,100
6,test,Wiki,gpt2,127,jwanders_text_1 vs jéské_couriano_text_4,100
7,test,Wiki,gpt2,128,jwanders_text_2 vs jéské_couriano_text_4,100
8,test,Wiki,gpt2,129,jwanders_text_5 vs jéské_couriano_text_4,100
